In [2]:
!pip install pandas openai matplotlib

In [ ]:
# ============================================================
# 1. Setup
# ============================================================

import os
import re
import time
import random
from pathlib import Path
from itertools import combinations
from datetime import datetime

import pandas as pd
from openai import OpenAI

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

BASELINE_RESPONSES_PATH = DATA_DIR / "baseline_model_responses.csv"
NEW_MODEL_RESPONSES_PATH = DATA_DIR / "new_model_responses.csv"

MERGED_RESPONSES_PATH = OUTPUT_DIR / "merged_responses.csv"
PAIRWISE_PATH = OUTPUT_DIR / "pairwise_comparisons.csv"
RAW_EVAL_PATH = OUTPUT_DIR / "gpt4o_raw_evaluations.csv"
PARSED_EVAL_PATH = OUTPUT_DIR / "parsed_evaluations.csv"

NEW_MODEL_NAME = "NEW_MODEL_NAME_HERE" # EDIT WITH YOU MODEL NAME

EVALUATION_MODE = "new_model_only"
# options:
# "new_model_only" = compare new model only against baseline models
# "full" = compare all models against each other

JUDGE_MODEL = "gpt-4o"
RANDOM_SEED = 42

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY is not None, "Please set OPENAI_API_KEY as an environment variable."

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# ============================================================
# 2. Load Existing Responses + New Model Responses
# ============================================================

baseline_df = pd.read_csv(BASELINE_RESPONSES_PATH)
new_model_df = pd.read_csv(NEW_MODEL_RESPONSES_PATH)

print("Baseline shape:", baseline_df.shape)
print("New model shape:", new_model_df.shape)

baseline_df.head()

In [ ]:
# ============================================================
# 3. Validate + Merge Input Files
# ============================================================

required_cols = {"id", "Prompt"}

missing_baseline = required_cols - set(baseline_df.columns)
missing_new = required_cols - set(new_model_df.columns)

assert not missing_baseline, f"Missing baseline columns: {missing_baseline}"
assert not missing_new, f"Missing new model columns: {missing_new}"

# new_model_responses.csv must have a column called exactly your NEW_MODEL_NAME
assert NEW_MODEL_NAME in new_model_df.columns, (
    f"NEW_MODEL_NAME='{NEW_MODEL_NAME}' not found in new_model_responses.csv. "
    f"Available columns: {list(new_model_df.columns)}"
)

# Keep only id, Prompt, and new model response
new_model_df = new_model_df[["id", "Prompt", NEW_MODEL_NAME]].copy()

# Check duplicate IDs
assert not baseline_df["id"].duplicated().any(), "baseline_model_responses.csv has duplicate ids."
assert not new_model_df["id"].duplicated().any(), "new_model_responses.csv has duplicate ids."

# Merge on id and Prompt
merged_df = baseline_df.merge(
    new_model_df,
    on=["id", "Prompt"],
    how="inner"
)

print("Merged shape:", merged_df.shape)

if len(merged_df) != len(baseline_df):
    print("Warning: some baseline prompts did not match the new model file.")

# Drop rows where new model response is missing
before = len(merged_df)
merged_df = merged_df.dropna(subset=[NEW_MODEL_NAME])
after = len(merged_df)

print(f"Dropped {before - after} rows with missing new model responses.")

merged_df.to_csv(MERGED_RESPONSES_PATH, index=False)
merged_df.head()

In [ ]:
# ============================================================
# 4. Create Pairwise Comparisons
# ============================================================

random.seed(RANDOM_SEED)

model_cols = [col for col in merged_df.columns if col not in ["id", "Prompt"]]

baseline_model_cols = [col for col in model_cols if col != NEW_MODEL_NAME]

if EVALUATION_MODE == "new_model_only":
    model_pairs = [(NEW_MODEL_NAME, model) for model in baseline_model_cols]
elif EVALUATION_MODE == "full":
    model_pairs = list(combinations(model_cols, 2))
else:
    raise ValueError("EVALUATION_MODE must be 'new_model_only' or 'full'.")

print("Number of model pairs:", len(model_pairs))
print("Total comparisons:", len(model_pairs) * len(merged_df))

rows = []

for _, row in merged_df.iterrows():
    for model1, model2 in model_pairs:
        response1 = row[model1]
        response2 = row[model2]

        # Randomize A/B position to reduce position bias
        if random.random() < 0.5:
            llm1, llm2 = model1, model2
            resp_a, resp_b = response1, response2
        else:
            llm1, llm2 = model2, model1
            resp_a, resp_b = response2, response1

        comparison_id = f"{row['id']}__{llm1}__vs__{llm2}"

        rows.append({
            "comparison_id": comparison_id,
            "id": row["id"],
            "Prompt": row["Prompt"],
            "LLM1": llm1,
            "LLM2": llm2,
            "LLM1 Response": resp_a,
            "LLM2 Response": resp_b,
        })

pairwise_df = pd.DataFrame(rows)
pairwise_df.to_csv(PAIRWISE_PATH, index=False)

pairwise_df.head()

In [ ]:
# ============================================================
# 5. GPT-4o Pairwise Evaluation Prompt
# ============================================================

SYSTEM_MESSAGE = "You are a helpful assistant that ranks models by the quality of their answers."

EVALUATION_PROMPT_TEMPLATE = """
Evaluate the following responses to the given prompt based on the criteria below. Provide your choices directly in the specified format without any additional explanations.

Prompt:
{prompt}

Response A:
{response_a}

Response B:
{response_b}

### Pairwise Comparison
Evaluate the responses according to these criteria and choose one of the following for each:
- A is better
- B is better
- Tie
- Both are bad

1. **Overall Quality:** Which response would a human reader prefer overall? (Consider clarity, coherence, and engagement.)
2. **Correctness:** Which response is more factually accurate and provides correct information?
3. **Relevance:** Which response better addresses the prompt and stays focused on the topic?

### Tunisian Arabic Usage Evaluation
Assign a score to each response based on the following scale:
- 0: No Tunisian Arabic used.
- 1: Some Tunisian Arabic used (e.g., mixed dialects, partial Tunisian expressions).
- 2: Fully in Tunisian Arabic (100%).

### Response Format:
Pairwise Comparison:
- Overall Quality: [A is better / B is better / Tie / Both are bad]
- Correctness: [A is better / B is better / Tie / Both are bad]
- Relevance: [A is better / B is better / Tie / Both are bad]

Tunisian Arabic Usage:
- Response A Score: [0 / 1 / 2]
- Response B Score: [0 / 1 / 2]
"""

In [ ]:
# ============================================================
# 6. Run GPT-4o Pairwise Evaluation
# ============================================================

def call_gpt4o_judge(prompt, response_a, response_b, max_retries=5):
    formatted_prompt = EVALUATION_PROMPT_TEMPLATE.format(
        prompt=prompt,
        response_a=response_a,
        response_b=response_b,
    )

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": formatted_prompt},
                ],
                temperature=0,
                max_tokens=512,
            )
            return completion.choices[0].message.content.strip()

        except Exception as e:
            wait_time = 2 ** attempt
            print(f"Error: {e}")
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

    return "Evaluation Failed"


# Load pairwise file
pairwise_df = pd.read_csv(PAIRWISE_PATH)

# Resume if raw evaluations already exist
if RAW_EVAL_PATH.exists():
    raw_eval_df = pd.read_csv(RAW_EVAL_PATH)
    done_ids = set(raw_eval_df["comparison_id"])
    print(f"Resuming from existing file. Completed: {len(done_ids)}")
else:
    raw_eval_df = pd.DataFrame()
    done_ids = set()

new_results = []

for idx, row in pairwise_df.iterrows():
    if row["comparison_id"] in done_ids:
        continue

    print(f"Evaluating {idx + 1}/{len(pairwise_df)}: {row['comparison_id']}")

    evaluation = call_gpt4o_judge(
        prompt=row["Prompt"],
        response_a=row["LLM1 Response"],
        response_b=row["LLM2 Response"],
    )

    result = row.to_dict()
    result["judge_model"] = JUDGE_MODEL
    result["judge_timestamp"] = datetime.now().isoformat()
    result["GPT-4o Evaluation"] = evaluation

    new_results.append(result)

    # Save checkpoint every 10 rows
    if len(new_results) % 10 == 0:
        checkpoint_df = pd.concat(
            [raw_eval_df, pd.DataFrame(new_results)],
            ignore_index=True
        )
        checkpoint_df.to_csv(RAW_EVAL_PATH, index=False)
        print(f"Checkpoint saved: {len(checkpoint_df)} rows")

# Final save
final_raw_df = pd.concat(
    [raw_eval_df, pd.DataFrame(new_results)],
    ignore_index=True
)

final_raw_df.to_csv(RAW_EVAL_PATH, index=False)
print(f"Saved raw evaluations to {RAW_EVAL_PATH}")

In [ ]:
# ============================================================
# 7. Parse GPT-4o Evaluations
# ============================================================

raw_df = pd.read_csv(RAW_EVAL_PATH)

QUALITY_PATTERN = r"Overall Quality:\s*(A is better|B is better|Tie|Both are bad)"
CORRECTNESS_PATTERN = r"Correctness:\s*(A is better|B is better|Tie|Both are bad)"
RELEVANCE_PATTERN = r"Relevance:\s*(A is better|B is better|Tie|Both are bad)"
A_USAGE_PATTERN = r"Response A Score:\s*([0-2])"
B_USAGE_PATTERN = r"Response B Score:\s*([0-2])"


def normalize_pairwise_choice(choice):
    if choice == "A is better":
        return "LLM1"
    if choice == "B is better":
        return "LLM2"
    if choice == "Tie":
        return "Tie"
    if choice == "Both are bad":
        return "Both are bad"
    return None


def extract_match(pattern, text):
    if pd.isna(text):
        return None
    match = re.search(pattern, str(text))
    return match.group(1) if match else None


def parse_evaluation(text):
    quality_raw = extract_match(QUALITY_PATTERN, text)
    correctness_raw = extract_match(CORRECTNESS_PATTERN, text)
    relevance_raw = extract_match(RELEVANCE_PATTERN, text)

    a_usage = extract_match(A_USAGE_PATTERN, text)
    b_usage = extract_match(B_USAGE_PATTERN, text)

    return pd.Series({
        "quality_raw": quality_raw,
        "correctness_raw": correctness_raw,
        "relevance_raw": relevance_raw,
        "quality_winner": normalize_pairwise_choice(quality_raw),
        "correctness_winner": normalize_pairwise_choice(correctness_raw),
        "relevance_winner": normalize_pairwise_choice(relevance_raw),
        "LLM1 Tunisian Usage Score": int(a_usage) if a_usage is not None else None,
        "LLM2 Tunisian Usage Score": int(b_usage) if b_usage is not None else None,
    })


parsed_cols = raw_df["GPT-4o Evaluation"].apply(parse_evaluation)
parsed_df = pd.concat([raw_df, parsed_cols], axis=1)

parsed_df.to_csv(PARSED_EVAL_PATH, index=False)

print("Parsing complete.")
print(parsed_df[[
    "quality_winner",
    "correctness_winner",
    "relevance_winner",
    "LLM1 Tunisian Usage Score",
    "LLM2 Tunisian Usage Score"
]].isna().sum())

parsed_df.head()

In [ ]:
# ============================================================
# 8. Inspect Failed or Unparsed Evaluations
# ============================================================

problem_rows = parsed_df[
    parsed_df["quality_winner"].isna()
    | parsed_df["correctness_winner"].isna()
    | parsed_df["relevance_winner"].isna()
    | parsed_df["LLM1 Tunisian Usage Score"].isna()
    | parsed_df["LLM2 Tunisian Usage Score"].isna()
]

print("Problem rows:", len(problem_rows))

problem_rows[[
    "comparison_id",
    "GPT-4o Evaluation",
    "quality_winner",
    "correctness_winner",
    "relevance_winner",
    "LLM1 Tunisian Usage Score",
    "LLM2 Tunisian Usage Score"
]].head(20)

In [ ]:
# Optional: save failed rows separately for rerun
FAILED_EVAL_PATH = OUTPUT_DIR / "failed_or_unparsed_evaluations.csv"
problem_rows.to_csv(FAILED_EVAL_PATH, index=False)

In [ ]:
# ============================================================
# 9. Compute Elo Leaderboards
# ============================================================

def update_elo(rating_a, rating_b, outcome, k=32, penalty=16):
    """
    outcome:
      1   = model A wins
      0.5 = tie
      0   = model B wins
      "bad" = both are bad, subtract penalty from both
    """
    if outcome == "bad":
        return rating_a - penalty, rating_b - penalty

    expected_a = 1 / (1 + 10 ** ((rating_b - rating_a) / 400))
    expected_b = 1 - expected_a

    new_rating_a = rating_a + k * (outcome - expected_a)
    new_rating_b = rating_b + k * ((1 - outcome) - expected_b)

    return new_rating_a, new_rating_b


def winner_to_outcome(winner):
    if winner == "LLM1":
        return 1
    if winner == "LLM2":
        return 0
    if winner == "Tie":
        return 0.5
    if winner == "Both are bad":
        return "bad"
    return None


def compute_elo_leaderboard(df, winner_col, initial_rating=1500, k=32, penalty=16):
    models = sorted(set(df["LLM1"]).union(set(df["LLM2"])))
    ratings = {model: initial_rating for model in models}

    valid_df = df.dropna(subset=[winner_col]).copy()

    for _, row in valid_df.iterrows():
        model_a = row["LLM1"]
        model_b = row["LLM2"]

        outcome = winner_to_outcome(row[winner_col])
        if outcome is None:
            continue

        ratings[model_a], ratings[model_b] = update_elo(
            ratings[model_a],
            ratings[model_b],
            outcome,
            k=k,
            penalty=penalty,
        )

    leaderboard = (
        pd.DataFrame(ratings.items(), columns=["LLM", "Elo"])
        .sort_values("Elo", ascending=False)
        .reset_index(drop=True)
    )
    leaderboard.insert(0, "Rank", range(1, len(leaderboard) + 1))

    return leaderboard


quality_leaderboard = compute_elo_leaderboard(parsed_df, "quality_winner")
correctness_leaderboard = compute_elo_leaderboard(parsed_df, "correctness_winner")
relevance_leaderboard = compute_elo_leaderboard(parsed_df, "relevance_winner")

quality_leaderboard.to_csv(OUTPUT_DIR / "leaderboard_quality.csv", index=False)
correctness_leaderboard.to_csv(OUTPUT_DIR / "leaderboard_correctness.csv", index=False)
relevance_leaderboard.to_csv(OUTPUT_DIR / "leaderboard_relevance.csv", index=False)

quality_leaderboard

In [ ]:
# ============================================================
# 10. Compute Tunisian Usage Leaderboard
# ============================================================

usage_rows = []

for _, row in parsed_df.iterrows():
    if not pd.isna(row["LLM1 Tunisian Usage Score"]):
        usage_rows.append({
            "LLM": row["LLM1"],
            "Tunisian Usage Score": row["LLM1 Tunisian Usage Score"],
        })

    if not pd.isna(row["LLM2 Tunisian Usage Score"]):
        usage_rows.append({
            "LLM": row["LLM2"],
            "Tunisian Usage Score": row["LLM2 Tunisian Usage Score"],
        })

usage_df = pd.DataFrame(usage_rows)

tunisian_usage_leaderboard = (
    usage_df
    .groupby("LLM", as_index=False)
    .agg(
        Avg_Tunisian_Usage_Score=("Tunisian Usage Score", "mean"),
        Count=("Tunisian Usage Score", "count"),
    )
    .sort_values("Avg_Tunisian_Usage_Score", ascending=False)
    .reset_index(drop=True)
)

tunisian_usage_leaderboard.insert(
    0,
    "Rank",
    range(1, len(tunisian_usage_leaderboard) + 1)
)

tunisian_usage_leaderboard.to_csv(
    OUTPUT_DIR / "leaderboard_tunisian_usage.csv",
    index=False
)


In [ ]:
# ============================================================
# 11. Combine Final Leaderboards into One Table
# ============================================================

final_summary = pd.DataFrame({
    "Rank": range(1, len(quality_leaderboard) + 1),
    "Quality": quality_leaderboard["LLM"],
    "Quality Elo": quality_leaderboard["Elo"].round(2),
    "Correctness": correctness_leaderboard["LLM"],
    "Correctness Elo": correctness_leaderboard["Elo"].round(2),
    "Relevance": relevance_leaderboard["LLM"],
    "Relevance Elo": relevance_leaderboard["Elo"].round(2),
    "Tunisian Usage": tunisian_usage_leaderboard["LLM"],
    "Avg Tunisian Usage Score": tunisian_usage_leaderboard["Avg_Tunisian_Usage_Score"].round(4),
})

final_summary.to_csv(OUTPUT_DIR / "final_leaderboard_summary.csv", index=False)


In [ ]:
# ============================================================
# 12. Optional: Plot Final Leaderboard Table
# ============================================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 6))
ax.axis("off")

table = ax.table(
    cellText=final_summary.values,
    colLabels=final_summary.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)

plt.savefig(OUTPUT_DIR / "final_leaderboard_summary.png", dpi=300, bbox_inches="tight")
plt.show()